# 02 - Modeling
## Step 1 - Baselines
- Goal: set the bar that XGBoost/LightGBM must beat. 
    - Two baselines: 
        - (1) always predict the average strength
        - (2) linear regression.

- Only `dev.csv` is used here. `test.csv` stays untouched until the very end.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold

from src.evaluate import regression_metrics
from src.features import DEV_PATH, FEATURES, TARGET

dev = pd.read_csv(DEV_PATH)
X, y, groups = dev[FEATURES], dev[TARGET], dev["recipe_id"]
print("dev->shape:", dev.shape," | X->shape:", X.shape, " | y->shape:", y.shape, " | groups->shape:", groups.shape)

dev->shape: (799, 10)  | X->shape: (799, 8)  | y->shape: (799,)  | groups->shape: (799,)


In [2]:
# 5-fold CV where each recipe stays inside one fold (same leakage rule as the dev/test split)
gkf = GroupKFold(n_splits=5)
rows = []
for fold, (tr, va) in enumerate(gkf.split(X, y, groups), start=1):
    y_tr, y_va = y.iloc[tr], y.iloc[va]
    preds = {
        "mean_predictor": np.full(len(va), y_tr.mean()),
        "linear_regression": LinearRegression().fit(X.iloc[tr], y_tr).predict(X.iloc[va]),
    }
    for name, p in preds.items():
        rows.append({"model": name, "fold": fold, **regression_metrics(y_va, p)})
cv = pd.DataFrame(rows)
cv.round(2)

,model,fold,rmse,mae,r2
0,mean_predictor,1,15.19,12.10,-0.00
1,linear_regression,1,10.04,7.87,0.56
2,mean_predictor,2,15.87,12.77,-0.00
3,linear_regression,2,10.36,8.30,0.57
4,mean_predictor,3,16.63,13.34,-0.00
5,linear_regression,3,11.45,9.13,0.53
6,mean_predictor,4,14.82,12.13,-0.01
7,linear_regression,4,9.25,7.41,0.61
8,mean_predictor,5,16.44,13.67,-0.00
9,linear_regression,5,10.56,8.44,0.59


In [3]:
# mean and spread across the 5 folds
cv.groupby("model")[["rmse", "mae", "r2"]].agg(["mean", "std"]).round(2)

rmse          mae          r2      
                    mean   std   mean   std  mean   std
model                                                  
linear_regression  10.33  0.80   8.23  0.65  0.57  0.03
mean_predictor     15.79  0.78  12.80  0.70 -0.00  0.00

## Step 2 - A fairer linear baseline
- Linear regression can only draw straight lines, but age gains strength along a curve. Before comparing with XGBoost, give the linear model two domain-driven features so the comparison is fair: `log_age` and `water_cement_ratio`. We add them one at a time to see which one matters.

In [4]:
from src.features import ENGINEERED, add_features

dev_fe = add_features(dev)

def cv_linear(cols):
    rows = []
    for tr, va in gkf.split(dev_fe, y, groups):
        model = LinearRegression().fit(dev_fe.iloc[tr][cols], y.iloc[tr])
        pred = model.predict(dev_fe.iloc[va][cols])
        rows.append(regression_metrics(y.iloc[va], pred))
    return pd.DataFrame(rows).mean()

variants = {
    "raw 8 features": FEATURES,
    "+ log_age": FEATURES + ["log_age"],
    "+ water_cement_ratio": FEATURES + ["water_cement_ratio"],
    "+ both": FEATURES + ENGINEERED,
}
pd.DataFrame({name: cv_linear(cols) for name, cols in variants.items()}).T.round(2)

,rmse,mae,r2
raw 8 features,10.33,8.23,0.57
+ log_age,7.08,5.55,0.80
+ water_cement_ratio,10.30,8.15,0.57
+ both,7.03,5.54,0.80


- **Findings (Step 2, linear regression)**:
    - `log_age` is the whole gain: R2 0.57 -> 0.80 and RMSE 10.3 -> 7.1 MPa. The age curve was what a straight line could not follow.
    - `water_cement_ratio` alone changes nothing (RMSE 10.33 -> 10.30). On top of `log_age` the change (7.08 -> 7.03) is within fold-to-fold noise (std about 0.8 for the raw model), so no claim of improvement. Kept in the feature list anyway; tree models and SHAP may use it differently.
    - Bar for the tree models: linear with both extra features, RMSE about 7.0, R2 about 0.80.

## Step 3 - Gradient-boosted trees (XGBoost, LightGBM)
These models build many small decision trees, each one correcting the errors of the previous ones. Trees can bend with age and combine features (cement with water with age) without us hand-making features. Default settings for now; tuning (Optuna) comes later. Each model is run on the raw 8 features and on all 10, to see whether the engineered features still matter for trees.

In [ ]:
def cv_model(make_model, cols):
    rows = []
    for tr, va in gkf.split(dev_fe, y, groups):
        model = make_model().fit(dev_fe.iloc[tr][cols], y.iloc[tr])
        pred = model.predict(dev_fe.iloc[va][cols])
        rows.append(regression_metrics(y.iloc[va], pred))
    r = pd.DataFrame(rows)
    return pd.Series({"rmse": r["rmse"].mean(), "rmse_std": r["rmse"].std(), "mae": r["mae"].mean(), "r2": r["r2"].mean()})

In [ ]:
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

ALL = FEATURES + ENGINEERED
models = {
    "linear (raw 8)": (LinearRegression, FEATURES),
    "linear (+both)": (LinearRegression, ALL),
    "xgboost (raw 8)": (lambda: XGBRegressor(random_state=42), FEATURES),
    "xgboost (+both)": (lambda: XGBRegressor(random_state=42), ALL),
    "lightgbm (raw 8)": (lambda: LGBMRegressor(random_state=42, verbose=-1), FEATURES),
    "lightgbm (+both)": (lambda: LGBMRegressor(random_state=42, verbose=-1), ALL),
}
pd.DataFrame({n: cv_model(f, c) for n, (f, c) in models.items()}).T.round(2)

## Step 3b - Which engineered features help the trees?
The table above only compared the raw 8 features against all 10, so it cannot say which feature (if any) helps. Here each feature is added alone, and every variant is compared with the raw 8 on the SAME folds. `diff_vs_raw` is the average change in RMSE (negative = better) and `better_in_folds` counts the folds (out of 5) where the variant beat the raw features.

In [ ]:
def fold_rmse(make_model, cols):
    out = []
    for tr, va in gkf.split(dev_fe, y, groups):
        model = make_model().fit(dev_fe.iloc[tr][cols], y.iloc[tr])
        pred = model.predict(dev_fe.iloc[va][cols])
        out.append(regression_metrics(y.iloc[va], pred)["rmse"])
    return np.array(out)

feature_sets = {
    "raw 8": FEATURES,
    "+ log_age": FEATURES + ["log_age"],
    "+ water_cement_ratio": FEATURES + ["water_cement_ratio"],
    "+ both": FEATURES + ENGINEERED,
}
tree_models = {
    "xgboost": lambda: XGBRegressor(random_state=42),
    "lightgbm": lambda: LGBMRegressor(random_state=42, verbose=-1),
}
rows = []
for m_name, make in tree_models.items():
    base = fold_rmse(make, feature_sets["raw 8"])
    for f_name, cols in feature_sets.items():
        r = fold_rmse(make, cols)
        rows.append({
            "model": m_name, "features": f_name, "rmse": r.mean(),
            "diff_vs_raw": (r - base).mean(), "better_in_folds": int((r < base).sum()),
        })
pd.DataFrame(rows).round(2)

- **Findings (Step 3b, feature ablation for trees)**:
    - `log_age` makes no difference for trees (no change at two decimals, not better in any fold). Trees split on thresholds, so a monotone transform like log leaves the splits unchanged.
    - `water_cement_ratio` has no reliable effect: XGBoost gets slightly worse (+0.05, better in 2 of 5 folds), LightGBM slightly better (-0.05, better in 4 of 5). The effects are tiny, opposite in sign, and far smaller than the fold-to-fold spread (about 1.0).
    - Decision: tree models use the raw 8 features. This keeps the API simple (no feature computation at prediction time). The engineered features stay only for the linear baseline.
    - Rule used: keep a feature only if it helps beyond noise for the model we ship.

## Step 4 - Hyperparameter tuning with Optuna
- Goal: find better settings (number of trees, learning rate, tree depth, ...) for XGBoost and LightGBM. Optuna tries 50 combinations per model, scoring each with the same grouped 5-fold CV on dev, and learns from earlier trials where to look next. The number of folds stays fixed at 5 because it is how we measure, not a model setting.

- Caveat: the best CV score found this way is slightly optimistic, because we picked the settings that scored best on these very folds. The honest number comes from `test.csv`, opened once at the end.

In [ ]:
from src.train import cv_rmse, tune

X8 = dev_fe[FEATURES]  # raw 8 features, as decided above
studies, results = {}, {}
for name in ["xgboost", "lightgbm"]:
    studies[name] = tune(name, X8, y, groups, n_trials=50)
    default = cv_rmse(tree_models[name], X8, y, groups).mean()
    results[name] = {"default_rmse": default, "tuned_rmse": studies[name].best_value}
    print(name, studies[name].best_params)
pd.DataFrame(results).T.round(2)